# 通用函数与向量化

学习目标：用通用函数完成批量数值计算，控制输出位置与计算类型，并区分逐元素运算、归约和累计。

前置知识：数组形状与轴、广播、dtype 与类型转换、布尔掩码、Python 函数。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的数据，后续单元沿用首次导入的 np。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 批量计算

把一组模拟温度统一上调 0.5 °C，可以逐项循环，也可以直接对整个数组相加。用数组操作表达批量计算，称为向量化（vectorization）。

下面两种写法处理同一组输入，先核对结果，再观察数组写法。

In [1]:
import numpy as np

temperatures = np.array([18.0, 20.0, 22.0])
loop_values = []
for value in temperatures:
    loop_values.append(value + 0.5)

adjusted = temperatures + 0.5
print(loop_values)  # 三个值依次为 18.5、20.5、22.5。
print(adjusted)  # 预期：[18.5 20.5 22.5]。
print(np.all(adjusted == np.array(loop_values)))  # 预期：True，本例精确可比。
print(adjusted.shape, adjusted.dtype)  # 预期：(3,) float64。
print(temperatures)  # 预期：[18. 20. 22.]，输入没有被修改。

[np.float64(18.5), np.float64(20.5), np.float64(22.5)]
[18.5 20.5 22.5]
True
(3,) float64
[18. 20. 22.]


## 2 通用函数

通用函数（universal function，ufunc）对数组逐元素计算，并支持广播、类型转换和输出参数。np.add() 是加法 ufunc，通常可以写成 +；显式调用函数便于传入 out、where 和 dtype 等参数。

下面两个输入的形状均为 (3,)，分别表示温度和各位置的修正量。

In [2]:
temperatures = np.array([18.0, 20.0, 22.0])
offsets = np.array([0.5, 1.0, -0.5])

print(np.add(temperatures, offsets))  # 预期：[18.5 21. 21.5]。
print(temperatures + offsets)  # 与 np.add() 的结果相同。
print(isinstance(np.add, np.ufunc))  # 预期：True，add 是 ufunc 对象。

[18.5 21.  21.5]
[18.5 21.  21.5]
True


## 3 常用逐元素计算

### 3.1 算术运算

下表中的 left、right 表示两个数值数组，其形状需要相同或满足广播条件。以下示例均为同形数组，除数均非零。

| 函数 | 中文名称／含义 | 运算符写法 |
| --- | --- | --- |
| np.add(left, right) | 逐元素加法 | left + right |
| np.subtract(left, right) | 逐元素减法 | left - right |
| np.multiply(left, right) | 逐元素乘法 | left * right |
| np.divide(left, right) | 逐元素除法 | left / right |

In [3]:
left = np.array([4, 9, 16], dtype=np.int64)
right = np.array([2, 3, 4], dtype=np.int64)

print(np.add(left, right))  # 预期：[6 12 20]。
print(np.subtract(left, right))  # 预期：[2 6 12]。
print(np.multiply(left, right))  # 预期：[8 27 64]，不是矩阵乘法。
quotients = np.divide(left, right)
print(quotients)  # 预期：[2. 3. 4.]。
print(quotients.shape, quotients.dtype)  # 预期：(3,) float64。

[ 6 12 20]
[ 2  6 12]
[ 8 27 64]
[2. 3. 4.]
(3,) float64


### 3.2 指数与对数

np.exp() 计算以自然常数 e 为底的指数；np.log() 计算自然对数。e 约为 2.71828，NumPy 用 np.e 表示它。np.log2() 和 np.log10() 分别计算以 2 和 10 为底的对数。

本例对数输入均为正实数。浮点计算的往返结果可能有舍入误差，检查接近程度时使用 np.allclose()；本例小规模 float64 计算采用绝对容差 1e-12，相对容差设为 0。

In [4]:
exponents = np.array([0.0, 1.0, 2.0])
values = np.exp(exponents)
restored = np.log(values)

print(values)  # 约为 [1. 2.71828183 7.38905610]。
print(restored)  # 约为 [0. 1. 2.]。
print(np.allclose(restored, exponents, rtol=0, atol=1e-12))  # 预期：True。
print(np.log2(np.array([1.0, 2.0, 8.0])))  # 预期：[0. 1. 3.]。
print(np.log10(np.array([1.0, 10.0, 100.0])))  # 预期：[0. 1. 2.]。

[1.         2.71828183 7.3890561 ]
[0. 1. 2.]
True
[0. 1. 3.]
[0. 1. 2.]


### 3.3 三角函数与角度单位

np.sin()、np.cos()、np.tan() 分别逐元素计算正弦、余弦和正切，输入角度使用弧度。已有角度值时，可以先用 np.deg2rad() 将度转换为弧度。

下面使用 0°、30°、45°，正切输入避开了 90° 等不适合直接求有限值的位置。

In [5]:
angles_deg = np.array([0.0, 30.0, 45.0])
angles_rad = np.deg2rad(angles_deg)

print(angles_rad)  # 约为 [0. 0.52359878 0.78539816]，单位为弧度。
print(np.sin(angles_rad))  # 约为 [0. 0.5 0.70710678]。
print(np.cos(angles_rad))  # 约为 [1. 0.86602540 0.70710678]。
print(np.tan(angles_rad))  # 约为 [0. 0.57735027 1.]。
print(angles_rad.shape)  # 预期：(3,)，每个角度对应一个结果。

[0.         0.52359878 0.78539816]
[0.         0.5        0.70710678]
[1.         0.8660254  0.70710678]
[0.         0.57735027 1.        ]
(3,)


### 3.4 舍入与取整

不同函数采用不同取整方向；得到整数数值不代表 dtype 自动变成整数。

| 函数 | 中文名称／含义 |
| --- | --- |
| np.rint() | 舍入到最近整数，恰好处于中点时取偶数 |
| np.floor() | 向负无穷方向取整 |
| np.ceil() | 向正无穷方向取整 |
| np.trunc() | 向零方向截去小数部分 |
| np.round() | 按 decimals 指定的小数位数舍入 |

rint、floor、ceil、trunc 是 ufunc；round 是数组舍入函数，不能据此把所有 NumPy 函数都当成 ufunc。round 的浮点算法也可能有误差，不保证把任意小数变成精确的十进制数。

In [6]:
values = np.array([-2.5, -1.5, 1.5, 2.5])
rounded = np.rint(values)

print(rounded)  # 预期：[-2. -2. 2. 2.]，中点取最近偶数。
print(np.floor(values))  # 预期：[-3. -2. 1. 2.]。
print(np.ceil(values))  # 预期：[-2. -1. 2. 3.]。
print(np.trunc(values))  # 预期：[-2. -1. 1. 2.]。
print(rounded.dtype)  # 预期：float64，数值取整不等于类型转换。
print(np.round(np.array([1.25, 2.75]), decimals=1))  # 预期：[1.2 2.8]。

[-2. -2.  2.  2.]
[-3. -2.  1.  2.]
[-2. -1.  2.  3.]
[-2. -1.  1.  2.]
float64
[1.2 2.8]


## 4 控制输出与计算类型

### 4.1 out 指定输出数组

out 指定把计算结果写入哪个数组。目标形状必须能容纳结果，dtype 也要符合类型转换规则；函数仍会返回该输出数组。

需要保留输入时使用单独的输出数组，需要修改输入时可以把输入自身作为 out。

In [7]:
values = np.array([1.0, 2.0, 3.0])
output = np.zeros(3, dtype=np.float64)
returned = np.multiply(values, 2.0, out=output)

print(output)  # 预期：[2. 4. 6.]。
print(returned is output)  # 预期：True，返回的就是指定的输出对象。
print(values)  # 预期：[1. 2. 3.]，输入未变。

np.add(values, 0.5, out=values)
print(values)  # 预期：[1.5 2.5 3.5]，直接写入原数组。

[2. 4. 6.]
True
[1. 2. 3.]
[1.5 2.5 3.5]


### 4.2 where 控制计算位置

ufunc 的 where 参数只在条件为 True 的位置计算并写入；False 位置保留 out 中原有的值。它控制计算位置，np.where() 函数则从两种候选值中选择。

使用 where 时先初始化 out。省略 out 会新建未初始化的输出，False 位置不会自动填零，不能把其中的内容当作确定结果。

下面只对非零分母执行除法，其余位置保留本例约定的 -1.0。

In [8]:
numerators = np.array([6.0, 8.0, 10.0])
denominators = np.array([2.0, 0.0, 5.0])
ratios = np.full(3, -1.0, dtype=np.float64)

np.divide(numerators, denominators, out=ratios, where=denominators != 0)
print(ratios)  # 预期：[3. -1. 2.]，中间位置未执行除法，保留初值。
print(ratios.shape, ratios.dtype)  # 预期：(3,) float64。

[ 3. -1.  2.]
(3,) float64


### 4.3 dtype 指定计算类型

dtype 参数为当前 ufunc 选择计算与结果类型；它不会把原数组永久改成该类型。应在计算前选择足以容纳结果的类型，而不是等溢出后再扩大输出类型。

下面两个 uint8 输入相加，指定 uint16 后可以容纳 260。

In [9]:
left = np.array([250, 10], dtype=np.uint8)
right = np.array([10, 20], dtype=np.uint8)
result = np.add(left, right, dtype=np.uint16)

print(result)  # 预期：[260 30]。
print(result.dtype)  # 预期：uint16。
print(left.dtype, right.dtype)  # 预期：uint8 uint8，输入类型未变。

[260  30]
uint16
uint8 uint8


### 4.4 原地运算的类型限制

普通加法可以生成浮点结果；整数数组的 += 则要写回原整数存储，不能在默认转换规则下接收浮点结果。下面直接显示 TypeError；对应的具体异常为 UFuncTypeError。

需要保留小数时，先明确转换为浮点数组，再进行原地计算。

In [10]:
counts = np.array([1, 2, 3], dtype=np.int64)
print(counts + 0.5)  # 预期：[1.5 2.5 3.5]，这是新结果。

[1.5 2.5 3.5]


In [11]:
# 预期 UFuncTypeError（TypeError 的子类）：加法产生浮点结果，默认转换规则不允许写回 int64 数组。
counts += 0.5

UFuncTypeError: Cannot cast ufunc 'add' output from dtype('float64') to dtype('int64') with casting rule 'same_kind'

In [12]:
print(counts)  # 预期：[1 2 3]，失败后本例输入未变。
adjusted = counts.astype(np.float64)
adjusted += 0.5
print(adjusted, adjusted.dtype)  # 预期：[1.5 2.5 3.5] float64。

[1 2 3]
[1.5 2.5 3.5] float64


## 5 归约与累计

### 5.1 reduce 合并一个轴

归约（reduction）沿指定轴反复应用二元运算，把该轴的多个值合并。例如 np.add.reduce() 沿轴求和；默认 axis=0，本章示例显式写出轴。

下面 shape 为 (2, 3)，行表示两天，列表示三个站点，值表示各站点每天的观测次数。axis=0 合并两天，axis=1 合并三个站点。

In [13]:
counts = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int64)
by_station = np.add.reduce(counts, axis=0)
by_day = np.add.reduce(counts, axis=1)

print(by_station, by_station.shape)  # 预期：[5 7 9] (3,)，每个站点两天合计。
print(by_day, by_day.shape)  # 预期：[6 15] (2,)，每天三个站点合计。
print(np.add.reduce(counts, axis=None))  # 预期：21，合并所有轴。

[5 7 9] (3,)
[ 6 15] (2,)
21


### 5.2 accumulate 保留各步结果

累计（accumulation）保留沿轴推进时的每一步结果，因此输出形状与输入相同。np.add.accumulate() 计算累计和；默认也沿 axis=0。

下面沿时间轴累计每日观测次数，第一行是首日值，第二行是截至第二天的合计。

In [14]:
counts = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int64)
running = np.add.accumulate(counts, axis=0)

print(running)  # 预期：第一行 [1 2 3]，第二行 [5 7 9]。
print(running.shape, running.dtype)  # 预期：(2, 3) int64。
print(np.add.reduce(counts, axis=0))  # 预期：[5 7 9]，只保留各站点最终合计。

[[1 2 3]
 [5 7 9]]
(2, 3) int64
[5 7 9]


## 6 clip 限制数值范围

限幅表示把小于下界的值换成下界，把大于上界的值换成上界，区间内和端点上的值保持不变。np.clip() 可以一次处理整个数组。

clip 不检查下界是否小于上界。标量下界大于上界时，结果全部等于上界，因此上下界的含义需要由调用者确认。

In [15]:
values = np.array([-2.0, 0.0, 3.0, 5.0, 8.0])
limited = np.clip(values, 0.0, 5.0)

print(limited)  # 预期：[0. 0. 3. 5. 5.]，包含区间内、两端和越界值。
print(limited.shape, limited.dtype)  # 预期：(5,) float64。
print(np.clip(values, 5.0, 0.0))  # 预期：五个 0.0，下界大于上界时全部取上界。
print(values)  # 预期：仍为原输入，未指定 out 时没有原地修改。

[0. 0. 3. 5. 5.]
(5,) float64
[0. 0. 0. 0. 0.]
[-2.  0.  3.  5.  8.]


## 7 vectorize 的用途与边界

已有处理单个数值的 Python 函数时，np.vectorize() 可以让它接收数组。otypes 指定输出类型，下面指定 float。

vectorize 主要提供调用便利，内部本质上仍是循环，不保证提速。已有直接数组操作时，可以使用对应的数组表达式；不能仅凭“函数接收了数组”判断它是高性能实现。

In [16]:
def keep_nonnegative(value):
    """把负数换成零，保留非负数。"""
    return 0.0 if value < 0 else value


values = np.array([-2.0, 0.0, 3.0])
wrapped = np.vectorize(keep_nonnegative, otypes=[float])
wrapped_result = wrapped(values)
array_result = np.clip(values, 0.0, None)

print(wrapped_result)  # 预期：[0. 0. 3.]。
print(array_result)  # 预期：[0. 0. 3.]，上界 None 表示不限制上界。
print(np.all(wrapped_result == array_result))  # 预期：True，仅核对结果，不代表提速。
print(wrapped_result.shape, wrapped_result.dtype)  # 预期：(3,) float64。

[0. 0. 3.]
[0. 0. 3.]
True
(3,) float64


## 8 综合应用：计算并限制比值

两次观测、三个通道的模拟数值如下，两个输入形状均为 (2, 3)。只在分母非零时计算比值，分母为零的位置约定填 0.0；随后把比值限制在 0.0～2.0 内。

按给定规则生成结果后，与手算值逐项核对；这里使用的整数和半整数都可以精确表示。

In [17]:
numerators = np.array([[2.0, 9.0, 4.0], [6.0, -2.0, 5.0]])
denominators = np.array([[2.0, 3.0, 0.0], [4.0, 2.0, 5.0]])
ratios = np.zeros((2, 3), dtype=np.float64)

np.divide(numerators, denominators, out=ratios, where=denominators != 0)
print(ratios)  # 预期：第一行 [1. 3. 0.]，第二行 [1.5 -1. 1.]。
np.clip(ratios, 0.0, 2.0, out=ratios)
print(ratios)  # 预期：第一行 [1. 2. 0.]，第二行 [1.5 0. 1.]。
print(ratios.shape, ratios.dtype)  # 预期：(2, 3) float64。
expected = np.array([[1.0, 2.0, 0.0], [1.5, 0.0, 1.0]])
print(np.all(ratios == expected))  # 预期：True，与手算结果一致。

[[ 1.   3.   0. ]
 [ 1.5 -1.   1. ]]
[[1.  2.  0. ]
 [1.5 0.  1. ]]
(2, 3) float64
True


## 9 选学：转换规则与其他 ufunc 用法

### 9.1 casting 控制允许的转换

casting 指定允许哪类类型转换。safe 只允许按类型规则保留数值的转换；same_kind 还允许同一类内部转换，例如 float64 转为 float32。ufunc 默认采用 same_kind，因此它不等于“保证不损失精度”。

np.can_cast() 可以先检查两种 dtype 是否符合指定规则；它不检查一次数值计算是否会溢出。

In [18]:
values = np.array([1.0, 2.0], dtype=np.float64)
output = np.zeros(2, dtype=np.float32)

print(np.can_cast(np.float64, np.float32, casting="safe"))  # 预期：False。
print(np.can_cast(np.float64, np.float32, casting="same_kind"))  # 预期：True。

False
True


In [19]:
# 预期 UFuncTypeError（TypeError 的子类）：safe 转换规则不允许把 float64 结果写入 float32 输出。
np.add(values, values, out=output, casting="safe")

UFuncTypeError: Cannot cast ufunc 'add' output from dtype('float64') to dtype('float32') with casting rule 'safe'

In [20]:
np.add(values, values, out=output, casting="same_kind")
print(output, output.dtype)  # 预期：[2. 4.] float32，本例数值可精确表示。

[2. 4.] float32


### 9.2 reduceat 按分段起点归约

对一维数组，reduceat() 用整数序列指定每段起点。通常一段从当前起点开始，到下一个起点之前结束，最后一段到数组末尾。

起点需要在数组范围内；若当前起点不小于下一个起点，该段仅取当前起点的一个元素，而不是得到空段。

In [21]:
values = np.array([1, 2, 3, 4, 5], dtype=np.int64)

print(np.add.reduceat(values, [0, 2, 4]))  # 预期：[3 7 5]，三段分别为 [1 2]、[3 4]、[5]。
print(np.add.reduceat(values, [2, 1]))  # 预期：[3 14]；首段只取位置 2，末段取位置 1 至末尾。

[3 7 5]
[ 3 14]


### 9.3 outer 计算所有配对

二元 ufunc 的 outer() 把第一个输入的每个元素与第二个输入的每个元素配对计算。下面两个输入分别为形状 (2,) 和 (3,) 的一维数组，结果形状为 (2, 3)。

结果行对应第一个输入，列对应第二个输入；这是所有配对的计算，不是同位置逐元素计算。

In [22]:
left = np.array([1, 2])
right = np.array([10, 20, 30])
pair_sums = np.add.outer(left, right)

print(pair_sums)  # 预期：第一行 [11 21 31]，第二行 [12 22 32]。
print(pair_sums.shape)  # 预期：(2, 3)，共六个配对结果。

[[11 21 31]
 [12 22 32]]
(2, 3)


### 9.4 frompyfunc 包装 Python 函数

np.frompyfunc() 把 Python 函数包装为 ufunc，nin、nout 分别指定输入参数和返回对象的数量。它产生的数组使用 object dtype，不会自动变为普通数值 dtype。

下面仅展示包装接口和返回类型，不把包装行为当作数值加速。

In [23]:
def add_one(value):
    """返回输入值加一。"""
    return value + 1


wrapped = np.frompyfunc(add_one, 1, 1)
result = wrapped(np.array([1, 2, 3]))

print(isinstance(wrapped, np.ufunc))  # 预期：True。
print(result)  # 预期：[2 3 4]。
print(result.dtype)  # 预期：object，与普通整数 ufunc 的结果类型不同。

True
[2 3 4]
object


### 9.5 复数的共轭与模

np.conjugate() 将每个复数的虚部变号，得到共轭；np.absolute() 对复数返回模。下面使用实部与虚部组成 3、4、5 直角三角形的数值，便于核对模。

In [24]:
values = np.array([3 + 4j, -3 - 4j], dtype=np.complex128)
conjugates = np.conjugate(values)
magnitudes = np.absolute(values)

print(conjugates)  # 预期：[3.-4.j -3.+4.j]，实部保持不变。
print(magnitudes)  # 预期：[5. 5.]。
print(conjugates.dtype, magnitudes.dtype)  # 预期：complex128 float64。

[ 3.-4.j -3.+4.j]
[5. 5.]
complex128 float64


## 本章小结

（1）ufunc 逐元素计算，支持广播和输出参数。向量化表达批量操作，不能据此承诺固定性能收益。

（2）out 指定写入对象，where=False 的位置保留原输出值，因此条件计算需要预先初始化输出。

（3）dtype 影响计算类型；原地操作还要满足目标数组的类型限制，整数存储不能默认接收浮点结果。

（4）reduce 合并指定轴，accumulate 保留各步结果；应能解释二者的值和形状。

（5）clip 限制上下界，但不会替调用者检查上下界顺序；vectorize 提供调用便利，不保证提速。

## 练习

（1）把下面模拟读数统一乘以 2，再加 1。分别用短循环和数组表达式实现，比较值、shape 和 dtype。

In [25]:
values = np.array([1.0, 2.0, 4.0])

# 在此分别实现两种写法并比较。
# 检查：结果均为 [3. 5. 9.]，数组结果 shape 为 (3,)，dtype 为 float64。
# 说明哪一组操作可以直接用 multiply 和 add 表达。

（2）输出必须保留原输入长度，在分母非零的位置计算除法，其余位置保留 -99.0。使用预初始化的 out 和 ufunc 的 where 完成任务，并说明为什么不能省略 out。

In [26]:
numerators = np.array([8.0, 5.0, 9.0, 7.0])
denominators = np.array([2.0, 0.0, 3.0, 0.0])

# 在此初始化输出并进行条件除法，在注释中说明方法选择理由。
# 检查：输出 [4. -99. 3. -99.]，shape 为 (4,)，dtype 为 float64。
# 检查：两个输入均保持原值；未计算位置保留的是明确的初值。

（3）先预测下面两个结果的值和形状，再运行。分别说明每行如何参与计算，以及哪个结果保留中间步骤。

In [27]:
values = np.array([[2, 3, 4], [5, 6, 7]], dtype=np.int64)
reduced = np.add.reduce(values, axis=1)
accumulated = np.add.accumulate(values, axis=1)

# 先记录预测，再核对归约后剩余的轴和累计结果中的每一步。
print(reduced, reduced.shape)
print(accumulated, accumulated.shape)

[ 9 18] (2,)
[[ 2  5  9]
 [ 5 11 18]] (2, 3)


（4）将输入限幅到 0～10，并解释区间外和两个端点的结果。再把上下界颠倒，观察结果；最后写出你会在调用前检查的上下界条件。

In [28]:
values = np.array([-3, 0, 4, 10, 12], dtype=np.int64)

# 在此比较上下界为 (0, 10) 和 (10, 0) 的结果。
# 检查：正常结果为 [0 0 4 10 10]；颠倒时全部为 0，而不是自动报错。
# 两次结果 shape 均为 (5,)，输入保持原值；用注释说明调用前的检查条件。

### 重点练习提示

对应第（2）题。先独立完成，再按需要查看提示。

（1）where 为 False 的位置，应该从哪里取得题目要求的 -99.0？

（2）先创建与分子同形的浮点输出，再只允许非零分母位置写入它。

### 重点练习参考解析

对应第（2）题。

先用 full_like(numerators, -99.0) 初始化输出，再调用 divide，把该数组传入 out，并将 denominators != 0 传入 where。选中位置写入 8/2 和 9/3，其他位置保留初值，结果为 [4, -99, 3, -99]，形状 (4,)，dtype 为 float64。

where 为 False 不会自动生成约定的缺失标记；省略 out 时，这些位置可能保留未初始化内容。也不能把任一输入直接作为 out，否则会改变本题要求保留的输入。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | 机制与参数：[Universal functions](https://numpy.org/doc/2.5/reference/ufuncs.html) 的定义、Optional keyword arguments（out、where、dtype、casting）、Methods 和 Available ufuncs；[Quickstart — Basic operations](https://numpy.org/doc/2.5/user/quickstart.html#basic-operations) 的数组计算与原地类型限制；[add](https://numpy.org/doc/2.5/reference/generated/numpy.add.html)、[divide](https://numpy.org/doc/2.5/reference/generated/numpy.divide.html) 的参数与返回值。数学函数：[exp](https://numpy.org/doc/2.5/reference/generated/numpy.exp.html)、[log](https://numpy.org/doc/2.5/reference/generated/numpy.log.html) 的含义、Notes 与示例；[sin](https://numpy.org/doc/2.5/reference/generated/numpy.sin.html) 的弧度输入，Universal functions 的 Trigonometric functions 小节；[rint](https://numpy.org/doc/2.5/reference/generated/numpy.rint.html) 的中点取偶数与输出类型；[floor](https://numpy.org/doc/2.5/reference/generated/numpy.floor.html)、[ceil](https://numpy.org/doc/2.5/reference/generated/numpy.ceil.html)、[trunc](https://numpy.org/doc/2.5/reference/generated/numpy.trunc.html) 的定义；[round](https://numpy.org/doc/2.5/reference/generated/numpy.round.html) 的 decimals 和浮点误差 Notes；[allclose](https://numpy.org/doc/2.5/reference/generated/numpy.allclose.html) 的 atol、rtol 与比较条件。归约和批量处理：[reduce](https://numpy.org/doc/2.5/reference/generated/numpy.ufunc.reduce.html)、[accumulate](https://numpy.org/doc/2.5/reference/generated/numpy.ufunc.accumulate.html) 的 axis、返回结果和二维示例；[clip](https://numpy.org/doc/2.5/reference/generated/numpy.clip.html) 的 out、None 边界与下界大于上界的 Notes；[vectorize](https://numpy.org/doc/2.5/reference/generated/numpy.vectorize.html) 的 otypes 与性能 Notes。选学：[can_cast](https://numpy.org/doc/2.5/reference/generated/numpy.can_cast.html) 的 casting 规则；[reduceat](https://numpy.org/doc/2.5/reference/generated/numpy.ufunc.reduceat.html) 的分段规则与三个例外；[outer](https://numpy.org/doc/2.5/reference/generated/numpy.ufunc.outer.html) 的配对和形状；[frompyfunc](https://numpy.org/doc/2.5/reference/generated/numpy.frompyfunc.html) 的 nin、nout 与 object 输出；[conjugate](https://numpy.org/doc/2.5/reference/generated/numpy.conjugate.html)、[absolute](https://numpy.org/doc/2.5/reference/generated/numpy.absolute.html) 的复数共轭与模。 |